In [ ]:
import pandas as pd

movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

print(movies.head())
print(ratings.head())

In [ ]:
# Merge datasets
data = pd.merge(ratings, movies, on='movieId')

# Check result
print(data.head())

In [ ]:
# Create user-movie matrix
movie_matrix = data.pivot_table(index='userId', columns='title', values='rating')

# Check matrix
print(movie_matrix.head())

In [ ]:
toy_story_ratings = movie_matrix['Toy Story (1995)']

print(toy_story_ratings.head())

In [ ]:
similar_movies = movie_matrix.corrwith(toy_story_ratings)

print(similar_movies.head())

In [ ]:
ratings_count = data.groupby('title')['rating'].count()
print(ratings_count.head())

In [ ]:
corr_df = pd.DataFrame(similar_movies, columns=['Correlation'])
corr_df.dropna(inplace=True)

In [ ]:
corr_df = corr_df.join(ratings_count)
corr_df.columns = ['Correlation', 'Num_Ratings']

In [ ]:
filtered = corr_df[corr_df['Num_Ratings'] > 50]

print(filtered.sort_values('Correlation', ascending=False).head(10))

In [ ]:
def recommend(movie_name, min_ratings=50):
    if movie_name not in movie_matrix.columns:
        return "Movie not found!"
    
    movie_ratings = movie_matrix[movie_name]
    similar_movies = movie_matrix.corrwith(movie_ratings)
    
    corr_df = pd.DataFrame(similar_movies, columns=['Correlation'])
    corr_df.dropna(inplace=True)
    
    corr_df = corr_df.join(ratings_count)
    corr_df.columns = ['Correlation', 'Num_Ratings']
    
    filtered = corr_df[corr_df['Num_Ratings'] > min_ratings]
    
    recommendations = filtered.sort_values('Correlation', ascending=False)
    
    return recommendations.iloc[1:11]

In [ ]:
recommend("Inception (2010)")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Replace NaN with 0
movie_matrix_filled = movie_matrix.fillna(0)

# Compute similarity between movies
cosine_sim = cosine_similarity(movie_matrix_filled.T)

# Convert to DataFrame
cosine_df = pd.DataFrame(cosine_sim, 
                         index=movie_matrix.columns, 
                         columns=movie_matrix.columns)

print(cosine_df.head())

In [ ]:
def recommend_cosine(movie_name):
    if movie_name not in cosine_df.columns:
        return "Movie not found!"
    
    similar_scores = cosine_df[movie_name]
    
    # Sort
    similar_movies = similar_scores.sort_values(ascending=False)
    
    return similar_movies.iloc[1:11]

In [ ]:
recommend_cosine("Toy Story (1995)")

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Fill NaN with 0
matrix = movie_matrix.fillna(0)

# Apply SVD
svd = TruncatedSVD(n_components=20)
matrix_reduced = svd.fit_transform(matrix)

# Reconstruct
matrix_approx = svd.inverse_transform(matrix_reduced)

# Convert to DataFrame
predictions = pd.DataFrame(matrix_approx, 
                           index=matrix.index, 
                           columns=matrix.columns)

print(predictions.head())

In [ ]:
def recommend_svd(user_id, n=10):
    user_ratings = predictions.loc[user_id]
    
    # Movies already rated
    already_rated = movie_matrix.loc[user_id].dropna().index
    
    # Remove rated movies
    recommendations = user_ratings.drop(already_rated)
    
    return recommendations.sort_values(ascending=False).head(n)

In [ ]:
recommend_svd(1)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use genres
cv = CountVectorizer(tokenizer=lambda x: x.split('|'))

genre_matrix = cv.fit_transform(movies['genres'])

# Compute similarity
genre_sim = cosine_similarity(genre_matrix)

# Convert to DataFrame
genre_df = pd.DataFrame(genre_sim, 
                        index=movies['title'], 
                        columns=movies['title'])

print(genre_df.head())

In [ ]:
def recommend_content(movie_name):
    if movie_name not in genre_df.columns:
        return "Movie not found!"
    
    similar_scores = genre_df[movie_name]
    
    return similar_scores.sort_values(ascending=False).iloc[1:11]

In [ ]:
recommend_content("Toy Story (1995)")

In [ ]:
def recommend_hybrid(movie_name, weight_collab=0.7, weight_content=0.3):
    
    if movie_name not in cosine_df.columns:
        return "Movie not found!"
    
    # Get scores
    collab_scores = cosine_df[movie_name]
    content_scores = genre_df[movie_name]
    
    # Combine scores
    hybrid_scores = (weight_collab * collab_scores) + (weight_content * content_scores)
    
    # Sort
    recommendations = hybrid_scores.sort_values(ascending=False)
    
    return recommendations.iloc[1:11]

In [ ]:
recommend_hybrid("Toy Story (1995)")

In [33]:
import pickle

pickle.dump(cosine_df, open('cosine.pkl', 'wb'))
pickle.dump(genre_df, open('genre.pkl', 'wb'))